## Setup
Ejecuta primero la celda de instalacion para tener todas las dependencias listas.

In [1]:
# Instalacion de dependencias necesarias para la practica
%pip install -q openai azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Práctica: Prompt Engineering y Parametrización de Modelos

Esta práctica tiene como objetivo experimentar de forma práctica con técnicas de prompt engineering y parámetros de modelos de IA generativa. Se divide en **2 partes independientes**, cada una entregable como un notebook `.ipynb` separado.

**🎯 Objetivo general:** Comprender mediante experimentación cómo diferentes técnicas de prompting y configuraciones de parámetros afectan las respuestas de modelos IAG, desarrollando criterio para aplicarlas en casos reales.

---


In [1]:
import os
from pathlib import Path

env_path = Path("../.env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value
    print("Variables cargadas desde .env")
else:
    print("No se encontró .env; usando variables ya definidas en el sistema")

Variables cargadas desde .env


## 1) Exploración de Técnicas de Prompt Engineering

**Objetivo:** Experimentar con técnicas de prompt engineering, entendiendo su propósito y efectividad en distintos contextos.


### 1.1 - Selección y Prueba de Técnicas
Debes **seleccionar al menos 6 técnicas** de las documentadas en `TEORIA.md` (las que te resulten más interesantes o útiles). Si quieres probar más de 6, adelante.

Para cada técnica seleccionada:
- **Explica** qué es la técnica **con tus propias palabras** (no copies la definición de la teoría)
- **Diseña un ejemplo práctico** usando Azure AI Foundry u OpenAI API
- **Ejecuta el prompt** y muestra la respuesta del modelo
- **Analiza el resultado**: ¿funcionó como esperabas? ¿qué ventajas observaste? ¿en qué casos usarías esta técnica?

**Documentación:** Consulta el archivo `TEORIA.md` de este mismo directorio para entender cada técnica.

### Rol/Persona
Le da un rol al modelo para que se comporte de cierta manera.


In [ ]:

import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

# Build the base URL: project_endpoint + /openai/v1 (no api-version needed)
base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input="Eres entrenador del Atletico de Madrid, ¿cómo plantearías el partido contra el" + 
    " FC Barcelona para aguantar el resultado de la vuelta de champions ganando 2-0?",
)

print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")


Response: Para plantear un partido contra el FC Barcelona tras una ventaja de 2-0 en la ida, la estrategia debe centrarse en mantener la solidez defensiva y aprovechar las oportunidades de contraataque. Aquí tienes un esquema para el planteamiento:

### 1. **Defensiva Sólida**
   - **Sistema Táctico:** Utiliza un 4-4-2 o un 4-2-3-1, dependiendo de la disponibilidad de jugadores y el plan para el medio campo. Ambos sistemas proporcionan solidez defensiva.
   - **Defensores Compactos:** Mantén la línea defensiva muy junta, minimizando los espacios entre los centrales y laterales. La comunicación será clave.
   - **Presión Moderada:** Presiona a los defensores del Barça cuando tengan el balón, pero no tan agresivo que dejes espacios atrás.

### 2. **Control del Medio Campo**
   - **Tácticas de Contención:** Utiliza a mediocampistas que puedan destruir el juego del Barcelona, como un pivote defensivo sólido que recupere balones.
   - **Ruptura de Juego:** Si el Barcelona tiene el control, 

### Especificidad (be explicit)
Darle un prompt directo y claro


In [ ]:

import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

# Build the base URL: project_endpoint + /openai/v1 (no api-version needed)
base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input="Resume en 5 líneas el viaje de Artemis II",
)

print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")


Response: Artemis II será la primera misión tripulada del programa Artemis de la NASA, prevista para lanzarse en 2024. Su objetivo principal es orbitar la Luna y probar sistemas vitales para futuras misiones a Marte. La tripulación, compuesta por cuatro astronautas, pasará aproximadamente diez días en el espacio. Durante la misión, se realizarán experimentos y evaluaciones en condiciones de microgravedad. Artemis II marca un paso crucial hacia el establecimiento de una presencia humana sostenida en la Luna y más allá.
Status:   completed
Output tokens: 108


### Few-shot (ejemplos en prompt)
Darle al modelo ejemplos para que sepa como tiene que comportarse o que tiene q devolver


In [6]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

prompt = '''Extrae el nombre y la ciudad de la siguiente frase y devuelve el resultado estrictamente en formato JSON.

Ejemplos:
- "Carlos vive en Madrid desde hace cinco años." -> {"nombre": "Carlos", "ciudad": "Madrid"}
- "Ayer me encontre con Ana paseando por Bogota." -> {"nombre": "Ana", "ciudad": "Bogota"}
- "El vuelo de Martin llega a Buenos Aires manana." -> {"nombre": "Martin", "ciudad": "Buenos Aires"}

Ahora hazlo con esta frase:
"Pedro se mudo recientemente a Barcelona por motivos de trabajo."'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input=prompt,
)


print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")

Response: ```json
{"nombre": "Pedro", "ciudad": "Barcelona"}
```
Status:   completed
Output tokens: 18


### Zero-shot con instrucción clara
Decirle al modelo que hacer sin ejemplos


In [7]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

prompt = '''Extrae el nombre y la ciudad de la siguiente frase y devuelve el resultado estrictamente en formato JSON.
"Pedro se mudo recientemente a Barcelona por motivos de trabajo."'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input=prompt,
)


print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")

Response: ```json
{
  "nombre": "Pedro",
  "ciudad": "Barcelona"
}
```
Status:   completed
Output tokens: 22


### Cues / priming del output
Darle una guía completa al modelo para tener la respuesta que se quiere

In [9]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

prompt = '''"Analiza la siguiente reseña de un cliente y extrae la queja principal y el tono del mensaje.
        Reseña: 'Llevo esperando mi paquete tres semanas. Llamo a atención al cliente y nadie me da una solución, es una vergüenza el servicio que dais.'
        Análisis:
        **Queja principal:**"'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input=prompt,
)


print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")


Response: **Queja principal:** La demora en la entrega del paquete y la falta de respuesta por parte del servicio de atención al cliente.

**Tono del mensaje:** Enfadado y frustrado.
Status:   completed
Output tokens: 40


### Constraint / Format forcing
Forzar a que el modelo te devuelva un formato específico

In [15]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

prompt = '''"Lee el siguiente texto y extrae los nombres de los proyectos, sus responsables y el estado actual.

**Reglas estrictas:**
1. Devuelve los datos ÚNICAMENTE en una tabla Markdown.
2. Las columnas deben ser exactamente: 'Proyecto', 'Responsable', 'Estado'.
3. NO incluyas saludos, despedidas, confirmaciones ni ningún texto fuera de la tabla.

Texto: 'El proyecto Alpha está a cargo de Laura y ya fue completado. Por otro lado, la migración a la nube (proyecto Beta) la dirige Carlos y sigue en progreso. Finalmente, el rediseño web (Gamma) lo lleva Elena y está pausado.'"'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input=prompt,
)


print(f"Response: \n{response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")


Response: 
| Proyecto     | Responsable | Estado      |
|--------------|-------------|-------------|
| Alpha        | Laura       | Completado  |
| Beta         | Carlos      | En progreso  |
| Gamma        | Elena       | Pausado     |
Status:   completed
Output tokens: 52




### 1.2 - Aplicación de Técnicas en Casos Reales
Después de probar las técnicas individualmente, **elige 2 casos de uso reales** (por ejemplo: generación de documentación técnica, análisis de sentimiento, extracción de datos estructurados, etc.) y aplica las técnicas más apropiadas para cada caso. Justifica por qué elegiste esas técnicas.



### Caso de uso 1: Analisis de comentarios del clasico

**Objetivo del caso:**
Convertir un comentario narrativo de un partido en informacion estructurada (eventos, minuto, jugador y equipo) para facilitar su analisis.

**Tecnicas elegidas y justificacion:**
- **Rol/Persona:** define al modelo como analista deportivo, aportando contexto para interpretar mejor los eventos del partido y la terminologia futbolistica.
- **Constraint / Format forcing:** obliga a devolver la salida en una tabla Markdown con columnas fijas, lo que hace el resultado claro, comparable y reutilizable en informes o dashboards.

**Valor practico:**
Esta combinacion transforma texto libre en datos ordenados, reduciendo ambiguedad y mejorando la calidad de la informacion para su consumo posterior.

In [ ]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

prompt = '''
"Actúa como un analista de datos deportivos. Tu tarea es leer el comentario de un partido de fútbol 
y extraer los eventos clave.

**Reglas estrictas:**
1. Extrae únicamente: Goles, Tarjetas Amarillas, Tarjetas Rojas y Sustituciones.
2. Muestra los resultados en una tabla Markdown con las columnas: 'Minuto', 'Evento', 'Jugador', 'Equipo'.
3. NO añadas ningún texto introductorio ni conclusiones.

**Comentario del partido:** '¡Comienza el clásico! En el minuto 12, Lamine Yamala abre el marcador para 
el FC Barcelona con un tiro cruzado. El partido se calienta y en el minuto 43 Gavi recibe la tarjeta amarilla 
por una dura entrada. Ya en la segunda parte, el Barcelona mueve el banquillo en el minuto 60: entra Ferran Torres 
y se retira Lewandowski. Poco después, en el 75, roja directa para Rüdiger tras frenar un contraataque claro, primera
del partido para el equipo blanco'"
'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input=prompt,
)

print(f"Response: \n{response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")


Response: 
| Minuto | Evento          | Jugador           | Equipo           |
|--------|-----------------|-------------------|-------------------|
| 12     | Gol             | Lamine Yamala     | FC Barcelona      |
| 43     | Tarjeta Amarilla| Gavi              | FC Barcelona      |
| 60     | Sustitución     | Ferran Torres     | FC Barcelona      |
| 60     | Sustitución     | Lewandowski       | FC Barcelona      |
| 75     | Tarjeta Roja    | Rüdiger           | Real Madrid       |
Status:   completed
Output tokens: 119


### Caso de uso 2: Analisis de comentarios de videojuego

**Objetivo del caso:**
Clasificar automaticamente feedback de jugadores para convertir comentarios libres en tickets accionables para el equipo correcto.

**Tecnicas elegidas y justificacion:**
- **Few-shot:** define criterios de clasificacion con ejemplos reales, lo que reduce ambiguedad y mejora la consistencia del resultado.
- **Format forcing:** fuerza una salida estructurada y valida para integrarla directamente con flujos automáticos .

**Valor practico:**
Permite priorizar feedback de la comunidad de forma rapida y escalable, facilitando el triaje y la asignacion de tareas sin revision manual inicial.

In [23]:
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
if not project_endpoint:
    raise ValueError("Falta AZURE_FOUNDRY_PROJECT_ENDPOINT en el entorno (.env).")

base_url = project_endpoint.rstrip("/") + "/openai/v1"

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)

prompt = '''
"Analiza el comentario de este jugador. Quiero que pienses paso a paso: primero identifica la emoción dominante, luego busca si hay un argumento válido detrás de la queja, y finalmente clasifica el comentario como 'Tóxico', 'Queja Constructiva' o 'Sugerencia'.

Ejemplos:
- Comentario: 'Sois unos inútiles, el juego da asco y lo desinstalo ya.' 
  -> Paso 1: La emoción es ira pura. Paso 2: No hay ningún argumento o error específico mencionado. Paso 3: Clasificación: Tóxico.
- Comentario: 'Es frustrante que la nueva escopeta mate de un tiro desde tan lejos, arruina la diversión.' 
  -> Paso 1: La emoción es frustración. Paso 2: El argumento válido es el desbalance de daño y alcance del arma. Paso 3: Clasificación: Queja Constructiva.

Comentario a analizar: 'Madre mía, me paso media hora farmeando materiales para que un error del servidor me tire de la partida y pierda todo. Arreglen sus servidores por favor que es injugable así.'
  ->"
'''

chat_model = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
response = client.responses.create(
    model=chat_model,
    input=prompt,
)

print(f"Response: \n{response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")


Response: 
Vamos a analizar el comentario paso a paso.

**Paso 1: Identificar la emoción dominante.**  
La emoción dominante en este comentario es frustración. El jugador expresa su descontento por haber perdido tiempo y progreso debido a un error del servidor.

**Paso 2: Buscar un argumento válido detrás de la queja.**  
El argumento válido aquí es que el jugador menciona un problema específico: el error del servidor que causó que se desconectara de la partida y perdiera los materiales en los que había estado trabajando. Este es un aspecto que puede afectar la experiencia del usuario de manera significativa.

**Paso 3: Clasificación.**  
La queja se presenta de manera directa pero también pide una solución específica ("Arreglen sus servidores por favor"). Dado que el comentario se centra en un problema concreto y pide una mejora sin insultar, lo clasificaría como: Queja Constructiva.

En resumen:
- **Emoción:** Frustración
- **Argumento válido:** Error del servidor y pérdida de materi

### Entregable (Parte 1):
Notebook `.ipynb` que incluya:
- **Sección de configuración** (imports, credenciales/API keys, conexión al modelo)
- **Una sección por cada técnica probada** con:
  - Explicación de la técnica (con tus palabras)
  - Código del prompt
  - Salida del modelo
  - Análisis crítico del resultado
- **Sección de casos de uso reales** con aplicación práctica
- **Conclusiones personales**: ¿Qué técnicas te parecieron más útiles? ¿Cuáles usarías en proyectos futuros? (opcional)

---